In [ ]:
# run_pups_full_A.py
import os, re, pickle, numpy as np, pandas as pd, torch
from sklearn.metrics import roc_auc_score
from pups_inference import load_model, esm2_encode, DEVICE

BASE = "/mnt/volume6/czj/labLGN/LabLZ/data_preparation/"
SRC  = BASE + "cell2024_model_single_subst.csv"
LM   = "/mnt/volume6/czj/labLGN/LabLZ/pups_trial/real_landmark.npy"
OUT  = "/mnt/volume6/czj/labLGN/LabLZ/pups_trial/pups_full_delta.pkl"   # {KEY: float32[29]}
SAVE_EVERY = 20

def parse_pos(m):
    """Fetch the position from a mutation string like 'K222E'."""
    mt = re.match(r"^[A-Z](\d+)[A-Z]$", str(m).strip())
    return int(mt.group(1)) if mt else None

@torch.no_grad()
def probs(model, seq, lm_t, pos):
    X, xlen = esm2_encode(seq, pos=pos)
    _, multi = model.call_model(X, xlen, lm_t)
    return torch.sigmoid(multi).squeeze(0).cpu().numpy()          # (29,)

def main():
    df = pd.read_csv(SRC)
    
    df["KEY"] = df["Gene"].astype(str) + "||" + df["Variant"].astype(str)
    
    lm_t = torch.from_numpy(np.load(LM).astype(np.float32)).unsqueeze(0).to(DEVICE)
    
    model = load_model()
    
    store = pickle.load(open(OUT, "rb")) if os.path.exists(OUT) else {}
    todo = df[~df["KEY"].isin(store)]
    print(f"to do {len(todo)} / all {len(df)}")
    
    for i, (_, r) in enumerate(todo.iterrows()):
        wt, mt, pos = r.get("sequence"), r.get("mutant_sequence"), parse_pos(r.get("Mutation_used"))
        
        # Skip if either sequence is not a string or position is None
        if not (isinstance(wt, str) and isinstance(mt, str) and pos):
            store[r["KEY"]] = None
        else:
            store[r["KEY"]] = (probs(model, mt, lm_t, pos) - probs(model, wt, lm_t, pos)).astype(np.float32)
        
        if (i + 1) % SAVE_EVERY == 0 or (i + 1) == len(todo):
            with open(OUT + ".tmp", "wb") as f:
                pickle.dump(store, f)
            os.replace(OUT + ".tmp", OUT)
            print(f"Saved {i+1}/{len(todo)}")
    print(f"Done {sum(v is not None for v in store.values())}/{len(store)} → {OUT}")

    # AUROC(|Δ|->reloc) for all
    lab = df[df["label_5class"].notna() & df["label_5class"].ne("C_unknown")].copy()
    lab["reloc"] = (lab["label_5class"] != "C1_no_reloc").astype(int)
    lab["l1"] = lab["KEY"].map({k: float(np.abs(v).sum()) for k, v in store.items() if v is not None})
    lab = lab.dropna(subset=["l1"])
    if lab["reloc"].nunique() > 1:
        print(f"Full AUROC(|Δ|->reloc): {roc_auc_score(lab['reloc'], lab['l1']):.3f}  (n={len(lab)})")

if __name__ == "__main__":
    main()

/mnt/volume6/czj/PUPS
to do 2179 / all 2179


Loading weights:   0%|          | 0/534 [00:00<?, ?it/s]

[transformers] EsmModel LOAD REPORT from: /mnt/volume6/czj/labLGN/LabLZ/esm2_650M
Key                       | Status     | 
--------------------------+------------+-
lm_head.bias              | UNEXPECTED | 
lm_head.layer_norm.bias   | UNEXPECTED | 
lm_head.layer_norm.weight | UNEXPECTED | 
lm_head.dense.weight      | UNEXPECTED | 
lm_head.dense.bias        | UNEXPECTED | 
pooler.dense.weight       | MISSING    | 
pooler.dense.bias         | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Saved 20/2179
Saved 40/2179
Saved 60/2179
Saved 80/2179
Saved 100/2179
Saved 120/2179
Saved 140/2179
Saved 160/2179
Saved 180/2179
Saved 200/2179
Saved 220/2179
Saved 240/2179
Saved 260/2179
Saved 280/2179
Saved 300/2179
Saved 320/2179
Saved 340/2179
Saved 360/2179
Saved 380/2179
Saved 400/2179
Saved 420/2179
Saved 440/2179
Saved 460/2179
Saved 480/2179
Saved 500/2179
Saved 520/2179
Saved 540/2179
Saved 560/2179
Saved 580/2179
Saved 600/2179
Saved 620/2179
Saved 640/2179
Saved 660/2179
Saved 680/2179
Saved 700/2179
Saved 720/2179
Saved 740/2179
Saved 760/2179
Saved 780/2179
Saved 800/2179
Saved 820/2179
Saved 840/2179
Saved 860/2179
Saved 880/2179
Saved 900/2179
Saved 920/2179
Saved 940/2179
Saved 960/2179
Saved 980/2179
Saved 1000/2179
Saved 1020/2179
Saved 1040/2179
Saved 1060/2179
Saved 1080/2179
Saved 1100/2179
Saved 1120/2179
Saved 1140/2179
Saved 1160/2179
Saved 1180/2179
Saved 1200/2179
Saved 1220/2179
Saved 1240/2179
Saved 1260/2179
Saved 1280/2179
Saved 1300/2179
Saved 1320/21